# Analisis de Resultados HFL - RN

Notebook para analizar corridas guardadas en `hfl_v7-RN/Results/#N`.

Base esperada:
- Formato estandarizado desde `#3` en adelante.
- `ascon_metrics_gateway_*.csv`
- `ascon_metrics_server_*.csv`
- `model_metrics_gateway_*.csv`
- `global_weights_history_*.csv`

Metricas que resume:
- latencia por canal y operacion
- bytes utiles (`pt_bytes`), bytes cifrados (`enc_bytes`) y overhead
- accuracy/loss local por gateway
- accuracy/loss global por ronda
- duracion aproximada entre rondas
- distribucion de etiquetas observadas por gateway


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

MODEL_NAME = "RN"
RESULTS_ROOT = Path(r"C:/Users/VivoBook/Downloads/microproyecto2/microproyecto 2 porque wtf/src/hfl_v7-RN/Results")
ANALYSIS_ROOT = Path(r"C:/Users/VivoBook/Downloads/microproyecto2/microproyecto 2 porque wtf/src/Analisis de Modelos")
ATTEMPT_ID = 4  # Usa None para tomar el ultimo intento disponible
MIN_STANDARDIZED_ATTEMPT = 3
OUTPUT_DIR = ANALYSIS_ROOT / MODEL_NAME / "analysis_outputs"

attempt_pattern = re.compile(r"#(\d+)$")
attempt_dirs = sorted(
    [p for p in RESULTS_ROOT.iterdir() if p.is_dir() and attempt_pattern.match(p.name)],
    key=lambda p: int(attempt_pattern.match(p.name).group(1)),
)

if not attempt_dirs:
    raise FileNotFoundError(f"No se encontraron carpetas de intentos en {RESULTS_ROOT}")

available_attempts = [int(attempt_pattern.match(p.name).group(1)) for p in attempt_dirs]
eligible_attempts = [attempt for attempt in available_attempts if attempt >= MIN_STANDARDIZED_ATTEMPT]
if not eligible_attempts:
    raise ValueError(f"No hay intentos estandarizados desde #{MIN_STANDARDIZED_ATTEMPT}")

selected_attempt_id = max(eligible_attempts) if ATTEMPT_ID is None else ATTEMPT_ID
attempt_dir = next((p for p in attempt_dirs if int(attempt_pattern.match(p.name).group(1)) == selected_attempt_id), None)
if attempt_dir is None:
    raise FileNotFoundError(f"No existe la carpeta del intento #{selected_attempt_id}")

print("Modelo:", MODEL_NAME)
print("Resultados base:", RESULTS_ROOT)
print("Intentos encontrados:", available_attempts)
print("Intento analizado:", attempt_dir)


In [ ]:
def read_many(paths):
    frames = []
    for path in sorted(paths):
        frame = pd.read_csv(path)
        frame["source_file"] = path.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

gateway_transport = read_many(attempt_dir.glob("ascon_metrics_gateway_*.csv"))
server_transport = read_many(attempt_dir.glob("ascon_metrics_server_*.csv"))
model_metrics = read_many(attempt_dir.glob("model_metrics_gateway_*.csv"))
global_history = read_many(attempt_dir.glob("global_weights_history_*.csv"))

transport_frames = []
if not gateway_transport.empty:
    gateway_transport["scope"] = "gateway"
    transport_frames.append(gateway_transport)
if not server_transport.empty:
    server_transport["scope"] = "server"
    transport_frames.append(server_transport)
transport_df = pd.concat(transport_frames, ignore_index=True, sort=False) if transport_frames else pd.DataFrame()

for df, numeric_cols in [
    (transport_df, ["pt_bytes", "enc_bytes", "overhead_bytes", "elapsed_ms", "fl_round"]),
    (model_metrics, ["fl_round", "num_samples", "accuracy", "loss", "buffer_target"]),
    (global_history, ["round", "accuracy", "loss", "w3_mag", "w4_normal", "w4_brute", "w4_scan"]),
]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

if not transport_df.empty and {"pt_bytes", "overhead_bytes"}.issubset(transport_df.columns):
    transport_df["overhead_ratio_pct"] = np.where(
        transport_df["pt_bytes"].gt(0),
        (transport_df["overhead_bytes"] / transport_df["pt_bytes"]) * 100.0,
        np.nan,
    )

if not global_history.empty and "time" in global_history.columns:
    parsed_time = pd.to_datetime(global_history["time"], format="%H:%M:%S", errors="coerce")
    global_history["duration_since_prev_round_s"] = parsed_time.diff().dt.total_seconds()

file_inventory = pd.DataFrame(
    [
        {"file": path.name, "size_kb": round(path.stat().st_size / 1024, 2)}
        for path in sorted(attempt_dir.iterdir())
        if path.is_file()
    ]
)

display(Markdown("### Inventario de archivos del intento"))
display(file_inventory)


In [ ]:
overview_rows = [
    {"metric": "Intento analizado", "value": f"#{selected_attempt_id}"},
    {"metric": "Eventos de transporte", "value": len(transport_df)},
    {"metric": "Registros de entrenamiento local", "value": len(model_metrics)},
    {"metric": "Rondas globales", "value": len(global_history)},
]

if not global_history.empty:
    overview_rows.extend([
        {"metric": "Accuracy global final", "value": f"{global_history['accuracy'].iloc[-1] * 100:.2f}%"},
        {"metric": "Loss global final", "value": f"{global_history['loss'].iloc[-1]:.4f}"},
        {"metric": "Mejor accuracy global", "value": f"{global_history['accuracy'].max() * 100:.2f}%"},
        {"metric": "Menor loss global", "value": f"{global_history['loss'].min():.4f}"},
    ])

overview_df = pd.DataFrame(overview_rows)
display(Markdown("### Resumen ejecutivo"))
display(overview_df)

def q95(series):
    return series.quantile(0.95) if len(series) else np.nan

transport_summary = pd.DataFrame()
if not transport_df.empty:
    transport_summary = (
        transport_df
        .groupby(["scope", "device_suffix", "channel", "operation"], dropna=False)
        .agg(
            events=("elapsed_ms", "size"),
            avg_ms=("elapsed_ms", "mean"),
            median_ms=("elapsed_ms", "median"),
            p95_ms=("elapsed_ms", q95),
            avg_pt_bytes=("pt_bytes", "mean"),
            avg_enc_bytes=("enc_bytes", "mean"),
            avg_overhead_bytes=("overhead_bytes", "mean"),
            avg_overhead_ratio_pct=("overhead_ratio_pct", "mean"),
        )
        .reset_index()
        .sort_values(["scope", "device_suffix", "channel", "operation"])
    )

display(Markdown("### Resumen de transporte por canal"))
display(transport_summary)

local_train_summary = pd.DataFrame()
if not model_metrics.empty:
    local_train_summary = (
        model_metrics
        .groupby(["device_suffix", "stage"], dropna=False)
        .agg(
            rounds=("fl_round", "nunique"),
            total_samples=("num_samples", "sum"),
            mean_accuracy=("accuracy", "mean"),
            best_accuracy=("accuracy", "max"),
            last_accuracy=("accuracy", "last"),
            mean_loss=("loss", "mean"),
            best_loss=("loss", "min"),
            last_loss=("loss", "last"),
        )
        .reset_index()
        .sort_values(["device_suffix", "stage"])
    )

display(Markdown("### Resumen de entrenamiento local por gateway"))
display(local_train_summary)


In [ ]:
label_distribution = pd.DataFrame()
if not gateway_transport.empty and {"channel", "operation", "sample_label_name", "device_suffix"}.issubset(gateway_transport.columns):
    label_source = gateway_transport.query("channel == 'ESP32->RPi' and operation == 'decrypt'").copy()
    if not label_source.empty:
        label_distribution = (
            label_source
            .groupby(["device_suffix", "sample_label_name"])
            .size()
            .rename("events")
            .reset_index()
            .sort_values(["device_suffix", "events"], ascending=[True, False])
        )

round_timing = pd.DataFrame()
if not global_history.empty:
    round_timing = global_history[["round", "time", "duration_since_prev_round_s", "accuracy", "loss"]].copy()

display(Markdown("### Distribucion de etiquetas observadas por gateway"))
display(label_distribution)

display(Markdown("### Duracion aproximada entre rondas globales"))
display(round_timing)


In [ ]:
analysis_fig_1, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

if not global_history.empty:
    axes[0].plot(global_history["round"], global_history["accuracy"] * 100, marker="o", color="#10b981")
    axes[0].set_title("Accuracy global por ronda")
    axes[0].set_xlabel("Ronda")
    axes[0].set_ylabel("Accuracy (%)")

    axes[1].plot(global_history["round"], global_history["loss"], marker="o", color="#f43f5e")
    axes[1].set_title("Loss global por ronda")
    axes[1].set_xlabel("Ronda")
    axes[1].set_ylabel("Loss")

    axes[2].plot(global_history["round"], global_history["w3_mag"], marker="o", label="W3 general")
    axes[2].plot(global_history["round"], global_history["w4_normal"], marker="o", label="W4 normal")
    axes[2].plot(global_history["round"], global_history["w4_brute"], marker="o", label="W4 bruteforce")
    axes[2].plot(global_history["round"], global_history["w4_scan"], marker="o", label="W4 scan_A")
    axes[2].set_title("Magnitud de pesos por ronda")
    axes[2].set_xlabel("Ronda")
    axes[2].set_ylabel("Magnitud media abs")
    axes[2].legend(loc="best")
else:
    axes[0].set_visible(False)
    axes[1].set_visible(False)
    axes[2].set_visible(False)

if not model_metrics.empty:
    for gateway_id, subset in model_metrics.sort_values("fl_round").groupby("device_suffix"):
        axes[3].plot(subset["fl_round"], subset["accuracy"] * 100, marker="o", label=gateway_id)
    axes[3].set_title("Accuracy local por gateway")
    axes[3].set_xlabel("Ronda FL")
    axes[3].set_ylabel("Accuracy (%)")
    axes[3].legend(loc="best")
else:
    axes[3].set_visible(False)

plt.tight_layout()
plt.show()

analysis_fig_2, axes = plt.subplots(1, 2, figsize=(16, 5))

if not transport_summary.empty:
    latency_plot = transport_summary.copy()
    latency_plot["label"] = latency_plot["device_suffix"].fillna("?") + " | " + latency_plot["channel"] + " | " + latency_plot["operation"]
    axes[0].barh(latency_plot["label"], latency_plot["avg_ms"], color="#38bdf8")
    axes[0].set_title("Latencia promedio por canal")
    axes[0].set_xlabel("ms")
    axes[0].set_ylabel("Dispositivo | Canal | Operacion")

    overhead_plot = transport_summary.copy()
    overhead_plot["label"] = overhead_plot["device_suffix"].fillna("?") + " | " + overhead_plot["channel"]
    axes[1].barh(overhead_plot["label"], overhead_plot["avg_overhead_bytes"], color="#f59e0b")
    axes[1].set_title("Overhead promedio por canal")
    axes[1].set_xlabel("bytes")
    axes[1].set_ylabel("Dispositivo | Canal")
else:
    axes[0].set_visible(False)
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
export_dir = OUTPUT_DIR / f"attempt_{selected_attempt_id}"
export_dir.mkdir(parents=True, exist_ok=True)

file_inventory.to_csv(export_dir / "file_inventory.csv", index=False)
overview_df.to_csv(export_dir / "overview.csv", index=False)

if not transport_summary.empty:
    transport_summary.to_csv(export_dir / "transport_summary.csv", index=False)
if not local_train_summary.empty:
    local_train_summary.to_csv(export_dir / "local_train_summary.csv", index=False)
if not label_distribution.empty:
    label_distribution.to_csv(export_dir / "label_distribution.csv", index=False)
if not round_timing.empty:
    round_timing.to_csv(export_dir / "round_timing.csv", index=False)

analysis_fig_1.savefig(export_dir / "metricas_globales_y_locales.png", dpi=200, bbox_inches="tight")
analysis_fig_2.savefig(export_dir / "metricas_transporte.png", dpi=200, bbox_inches="tight")

print("Analisis exportado en:", export_dir)
